In [2]:
import glob
import pandas as pd

files = sorted(f for f in glob.glob("data/benchmark_*.csv") if "dataset" not in f)
model_dfs = {}
for path in files:
    tag = path.split("benchmark_")[1].replace(".csv", "")
    df = pd.read_csv(path, dtype=str).head(20)
    df["steps"] = pd.to_numeric(df["steps"], errors="coerce").fillna(0).astype(int)
    model_dfs[tag] = df

if not model_dfs:
    print("No benchmark result files found.")
else:
    base = next(iter(model_dfs.values()))[["product_description", "answer"]].copy()

    for tag, df in model_dfs.items():
        done = df["steps"] > 0
        base[f"{tag}_submitted"] = df["submitted"].where(done).fillna("-")
        base[f"{tag}_result"]    = df["correct"].where(done).map({"True": "OK", "False": "FAIL"}).fillna("-")

    # Per-model accuracy summary
    print("=== Accuracy summary ===")
    for tag in model_dfs:
        total   = (base[f"{tag}_submitted"] != "-").sum()
        correct = (base[f"{tag}_result"] == "OK").sum()
        print(f"  {tag}: {correct}/{total} ({correct/total:.1%})" if total else f"  {tag}: 0 completed rows")
    print()

    # Table: only rows completed in at least one model
    has_run = pd.Series(False, index=base.index)
    for tag in model_dfs:
        has_run |= base[f"{tag}_submitted"] != "-"

    display_df = base[has_run].copy().reset_index(drop=True)
    display_df["product_description"] = display_df["product_description"].str[:55]

    pd.set_option("display.max_colwidth", 60)
    pd.set_option("display.max_rows", None)
    display(display_df)

=== Accuracy summary ===
  Qwen3-14B-MLX-4bit: 18/20 (90.0%)
  Qwen3-4B-MLX-4bit: 15/20 (75.0%)
  Qwen3-8B-MLX-4bit: 17/20 (85.0%)



,product_description,answer,Qwen3-14B-MLX-4bit_submitted,Qwen3-14B-MLX-4bit_result,Qwen3-4B-MLX-4bit_submitted,Qwen3-4B-MLX-4bit_result,Qwen3-8B-MLX-4bit_submitted,Qwen3-8B-MLX-4bit_result
0,Fresh globe artichokes (Cynara scolymus) packed in 5 kg,070991,070991,OK,070991,OK,070991,OK
1,"Live grass carp (Ctenopharyngodon idellus), farmed, int",030193,030193,OK,010690,FAIL,030193,OK
2,"Live abalone (Haliotis discus) in shell, packed in aera",030781,030781,OK,010690,FAIL,030781,OK
3,"Washed and disinfected duck down, bulk packed in plasti",050510,050510,OK,050510,OK,050510,OK
4,"Frozen shelled edamame (young soybeans), uncooked and p",071029,200490,FAIL,071022,FAIL,071029,OK
5,Fresh golden chanterelle mushrooms (Cantharellus cibari,070953,070953,OK,200390,FAIL,070953,OK
6,Certified organic durum wheat seeds intended for sowing,100111,100111,OK,100111,OK,100111,OK
7,Spray-dried standardized ginseng root extract (Panax gi,130219,130219,OK,130219,OK,210690,FAIL
8,Bulk shipment of crude low erucic acid rapeseed oil (ca,151411,151411,OK,151411,OK,151411,OK
9,"Canned mantis shrimp (Stomatopoda), shelled and preserv",160540,160540,OK,030619,FAIL,030699,FAIL


In [5]:
import json
import pandas as pd

MODEL = "Qwen3-8B-MLX-4bit"
INDEX = 12

path = f"data/benchmark_{MODEL}.csv"
df = pd.read_csv(path, dtype=str)
row = df.iloc[INDEX]

steps = int(float(row.get("steps") or 0))
if steps == 0:
    print(f"Row {INDEX} has not been completed yet (steps=0).")
else:
    print(f"Product : {row['product_description']}")
    print(f"Answer  : {row['answer']}")
    print(f"Submitted: {row['submitted']}")
    print(f"Correct : {row['correct']}")
    print(f"Reward  : {row['reward']}")
    print(f"Steps   : {steps}")
    print()

    messages = json.loads(row["messages"])
    for msg in messages:
        role = msg.get("role", "?").upper()
        sep = "─" * 60
        print(sep)
        print(f"[{role}]")

        tool_calls = msg.get("tool_calls")
        content = msg.get("content") or ""

        if role == "ASSISTANT":
            if content:
                print(content)
            if tool_calls:
                for tc in tool_calls:
                    fn = tc["function"]
                    args = json.loads(fn["arguments"]) if isinstance(fn["arguments"], str) else fn["arguments"]
                    print(f"  -> {fn['name']}({json.dumps(args)})")
        elif role == "TOOL":
            print(msg.get("content", ""))
        else:
            print(content)
    print("─" * 60)


Product : Tomato paste, concentrated and packed in industrial 10 kg aseptic bags, not containing whole or pieces of tomato, prepared otherwise than by vinegar or acetic acid
Answer  : 200290
Submitted: nan
Correct : False
Reward  : 0
Steps   : 1

────────────────────────────────────────────────────────────
[SYSTEM]
Determine the correct 6-digit code for a product description using only the provided tools.

The hierarchy is:
Section (capital letter) > 2-digit chapter > 4-digit heading > 6-digit subheading

General approach:
- Use the tools to navigate the hierarchy step by step.
- Consider only the codes returned by the tools at each level.
- Never anticipate, infer, invent, or guess codes that have not been explicitly returned by a tool.
- Do not rely on prior knowledge of the classification system; the tools are the sole source of truth.
- When moving to a deeper level, choose only among the candidates provided by the previous tool call.
- If the correct code cannot be determined from

In [ ]:
import pandas as pd
raise Exception('Carefull')
# ── Configuration ──────────────────────────────────────────
MODEL = "Qwen3-4B-MLX-4bit"
INDEX = 14
# ───────────────────────────────────────────────────────────

path = f"data/benchmark_{MODEL}.csv"
df = pd.read_csv(path, dtype=str)
row = df.iloc[INDEX]

print(f"Row {INDEX}: {row['product_description'][:80]}")
print(f"  submitted={row.get('submitted')}  correct={row.get('correct')}  steps={row.get('steps')}")
print()

for col in ("submitted", "reward", "correct", "steps", "messages"):
    df.at[INDEX, col] = None

df.to_csv(path, index=False)
print(f"Cleared row {INDEX} in {path} — will be recomputed on next benchmark run.")


Row 14: Battery-grade lithium hydroxide monohydrate (LiOH·H2O) powder, 99.5% purity, use
  submitted=nan  correct=False  steps=5

Cleared row 14 in data/benchmark_Qwen3-4B-MLX-4bit.csv — will be recomputed on next benchmark run.
